# 🎞️ ROTBOTS — Assemble
## Interleaved: AI + Archive + Upload → Final Video

---

Builds the final video following the storyboard scene order.

---

In [ ]:
# SETUP
import json, subprocess, shutil, os, re
from pathlib import Path
from IPython.display import display, Markdown, Video
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)
print('✅ Setup')

In [ ]:
# LOAD SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
if not sessions: print('❌ No plan!')
else:
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
with open(SESSION_DIR/'storyboard.json') as f: storyboard = json.load(f)
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'
sub_file = SESSION_DIR / 'subtitles.ass'
narration_file = AUDIO_DIR / 'narration_full.mp3' if AUDIO_DIR.exists() else None
has_narration_file = narration_file is not None and narration_file.exists()
essay = None
if (SESSION_DIR/'essay_script.json').exists():
    with open(SESSION_DIR/'essay_script.json') as f: essay = json.load(f)
archive_clips = []
if (SESSION_DIR/'archive_clips.json').exists():
    with open(SESSION_DIR/'archive_clips.json') as f: archive_clips = json.load(f).get('clips',[])
upload_clips = []
if (SESSION_DIR/'upload_clips.json').exists():
    with open(SESSION_DIR/'upload_clips.json') as f: upload_clips = json.load(f).get('clips',[])
prompts_data = []
if (SESSION_DIR/'prompts.json').exists():
    with open(SESSION_DIR/'prompts.json') as f: prompts_data = json.load(f)
effects_map = {int(p['scene']):p for p in prompts_data if p.get('ffmpeg_effects')}
ai_count = sum(1 for s in storyboard if s['scene_type']=='ai_generated')
arc_count = sum(1 for s in storyboard if s['scene_type']=='archive')
upl_count = sum(1 for s in storyboard if s['scene_type']=='upload')
print(f'✅ Session: {SESSION_NAME}')
print(f'   📋 {len(storyboard)} scenes: 🤖{ai_count} AI + 🏛️{arc_count} archive + 📂{upl_count} upload')
print(f'   🎙️ {len(audio_files)} narrations | 📝 Subs: {"✅" if sub_file.exists() else "—"} | 🎨 {len(effects_map)} effects')

In [ ]:
# OPTIONS
ENABLE_NARRATION = True
ENABLE_MUSIC = plan.get('enable_music', False)
ENABLE_SUBTITLES = plan.get('enable_subtitles', False)
ENABLE_CREDITS = plan.get('enable_credits', True)
MUSIC_PROMPT = 'Ambient cinematic background music'
MUSIC_VOLUME = 0.10
music_path = None; credits_path = None
print('🎬 Options:')
for n,v in [('Narration',ENABLE_NARRATION),('Music',ENABLE_MUSIC),('Subtitles',ENABLE_SUBTITLES),('Credits',ENABLE_CREDITS)]:
    print(f'   {n}: {"✅" if v else "❌"}')

In [ ]:
# HELPERS
def dur(p):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0
def ff(cmd,desc=''):
    if desc: print(f'   {desc}...',end=' ',flush=True)
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if r.returncode==0:
        if desc: print('✅')
        return True
    if desc: print('❌')
    print(f'   {r.stderr[-300:]}'); return False
def build_effect_filter(name, intensity=0.7):
    i=max(0.0,min(1.0,intensity))
    if name=='film_grain': return f'noise=alls={int(12*i)}:allf=t'
    if name=='vhs_artifacts': return f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)},eq=contrast={1+0.1*i:.3f}:brightness={0.02*i:.3f}:saturation={1-0.15*i:.3f}'
    if name=='celluloid_scratches': return f'noise=c0s={int(15*i)}:c0f=t:c1s={int(15*i)}:c1f=t,eq=contrast={1+0.05*i:.3f}:brightness={-0.01*i:.3f}'
    if name=='sepia_tone': inv=1-i; return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name=='bw_transition': inv=1-i; return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    if name=='color_grade_warm': return f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f},colorbalance=rs={0.1*i:.3f}:gs={0.05*i:.3f}:bs={-0.05*i:.3f}'
    if name=='color_grade_cool': return f'eq=saturation={1-0.1*i:.3f},colorbalance=rs={-0.05*i:.3f}:gs=0:bs={0.12*i:.3f}'
    if name=='vignette': return f'vignette=PI/4*{i:.3f}'
    if name=='flicker': return f"noise=alls={int(8*i)}:allf=t,eq=brightness='{0.02*i:.4f}*sin(2*PI*t*3)'"
    if name=='desaturate': return f'eq=saturation={0.5+0.5*(1-i):.3f}'
    return None

In [ ]:
# STEP 1: BUILD SEQUENCE FROM STORYBOARD
print(f'🔧 Step 1: Building {len(storyboard)} scenes from storyboard...')
norm=[]; arc_idx=0; upl_idx=0
for sc in storyboard:
    sn=sc['scene'];stype=sc['scene_type']
    icon='🤖' if stype=='ai_generated' else '🏛️' if stype=='archive' else '📂'
    out=TEMP/f'scene_{sn:03d}.mp4'; src=None
    if stype=='ai_generated':
        c=VIDEOS_DIR/f'scene_{sn:03d}.mp4'
        if c.exists(): src=c
        else: print(f'   {icon} Scene {sn}: ❌ missing (run 07)'); continue
    elif stype=='archive':
        if arc_idx<len(archive_clips): src=Path(archive_clips[arc_idx]['path']); arc_idx+=1
        if not src or not src.exists(): print(f'   {icon} Scene {sn}: ❌ missing (run 03)'); continue
    elif stype=='upload':
        if upl_idx<len(upload_clips): src=Path(upload_clips[upl_idx]['path']); upl_idx+=1
        if not src or not src.exists(): print(f'   {icon} Scene {sn}: ❌ missing (run 04)'); continue
    vf='scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    eff_tag=''
    if stype=='ai_generated' and sn in effects_map:
        p=effects_map[sn]
        for eff in p.get('ffmpeg_effects',[]):
            f=build_effect_filter(eff,p.get('effect_intensity',0.7))
            if f: vf+=','+f
        eff_tag=f' +{p.get("ffmpeg_effects",[])}'
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p','-c:v','libx264','-preset','fast','-crf','23','-an','-t',str(sc.get('duration',5)),str(out)],f'{icon} Scene {sn}{eff_tag}'):
        norm.append(out)
print(f'✅ {len(norm)}/{len(storyboard)} scenes')

In [ ]:
# STEP 2: MUSIC
if ENABLE_MUSIC:
    print('🎵 Step 2: Music...')
    try:
        import fal_client, time, requests
        if not os.environ.get('FAL_KEY'): raise Exception('No FAL_KEY')
        td=sum(dur(v) for v in norm)+10
        result=fal_client.subscribe('beatoven/music-generation',arguments={'prompt':MUSIC_PROMPT,'duration':min(150,int(td))})
        url=result.get('audio',{}).get('url') if isinstance(result.get('audio'),dict) else result.get('audio') or result.get('url','')
        if url: music_path=TEMP/'music.wav'; open(music_path,'wb').write(requests.get(url,timeout=120).content); print('   ✅')
    except Exception as e: print(f'   ❌ {e}')
else: print('🎵 Step 2: skipped')

In [ ]:
# STEP 3: CREDITS
if ENABLE_CREDITS:
    print('📜 Step 3: Credits...')
    title=essay.get('title','Untitled') if essay else 'Untitled'
    sources=essay.get('credits',{}).get('sources',[]) if essay else []
    lines=[title,'','Generated by ROTBOTS','','— Sources —']+[s[:60] for s in sources]+['','— Pipeline —','LLM: Groq','Video: fal.ai','Voice: Edge-TTS','FFmpeg','','LI-MA TDA 2026']
    D=8;LH=42;spd=(720+len(lines)*LH)/D
    flt=[f"drawtext=text='{l.replace(chr(39),chr(8217)).replace(chr(58),chr(92)+chr(58))}':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t" for i,l in enumerate(lines) if l]
    credits_path=TEMP/'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24','-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',str(credits_path)],'Credits')
else: print('📜 Step 3: skipped')

In [ ]:
# STEPS 4-6: CONCAT + AUDIO (with padding) + SUBTITLES
print('🎬 Step 4: Concat...')
cf=TEMP/'concat.txt'
with open(cf,'w') as f:
    for v in norm: f.write(f"file '{v}'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out=TEMP/'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)],'Concat')
video_duration = dur(concat_out)
print(f'   {video_duration:.1f}s')

audio_out=TEMP/'with_audio.mp4'
has_narr=ENABLE_NARRATION and has_narration_file
has_mus=music_path is not None and music_path.exists()
if has_narr:
    print('🎙️ Step 5: Audio...')
    # Concat all narration files
    acf=TEMP/'ac.txt'
    with open(acf,'w') as f:
        for a in audio_files: f.write(f"file '{a}'\n")
    narr=TEMP/'narration.mp3'
    ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(acf),'-c','copy',str(narr)],'Concat narration')
    
    # Pad narration with silence to match video length (fixes credits being cut off)
    narr_padded=TEMP/'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narr),'-af',f'apad=whole_dur={video_duration}','-c:a','libmp3lame','-b:a','128k',str(narr_padded)],'Pad audio to video length')
    
    # Mix with music if enabled
    audio_track = narr_padded
    if has_mus:
        mx=TEMP/'mixed.mp3'
        ff(['ffmpeg','-y','-i',str(narr_padded),'-i',str(music_path),'-filter_complex',f'[0:a]volume=1.0[n];[1:a]volume={MUSIC_VOLUME}[m];[n][m]amix=inputs=2:duration=first[out]','-map','[out]','-c:a','libmp3lame',str(mx)],'Mix narration + music')
        audio_track = mx
    
    # Merge audio with video (no -shortest needed since audio is already padded)
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(audio_track),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)],'Merge audio + video')
elif has_mus:
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(music_path),'-c:v','copy','-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0','-shortest',str(audio_out)],'Music only')
else: shutil.copy2(str(concat_out),str(audio_out))

# Subtitles
final=SESSION_DIR/'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    print('📝 Step 6: Subtitles...')
    la=TEMP/'subtitles.ass';shutil.copy2(str(sub_file),str(la))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs (ass)'):
        if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'subtitles={str(la)}','-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)],'Burn subs (subtitles)'):
            shutil.copy2(str(audio_out),str(final))
else: shutil.copy2(str(audio_out),str(final))
if final.exists(): print(f'\n✅ Final: {dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')

---
## 🎬 Watch!

In [ ]:
title=essay.get('title','Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# 🎬 {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final),embed=True,width=720))
    except: print(f'Drive: rotbots_workshop/{SESSION_NAME}/final_video.mp4')
else: print('❌')

In [ ]:
# SUMMARY
print('📊 Pipeline Summary')
print('='*50)
st=[]
st.append(f'🤖 {ai_count} AI scenes')
if arc_count: st.append(f'🏛️ {arc_count} archive scenes')
if upl_count: st.append(f'📂 {upl_count} upload scenes')
if effects_map: st.append(f'🎨 {len(effects_map)} effects')
st.append(f'🎙️ {len(audio_files)} narrations')
if ENABLE_MUSIC and music_path: st.append('🎵 Music')
if ENABLE_SUBTITLES and sub_file.exists(): st.append('📝 Subtitles')
if ENABLE_CREDITS: st.append('📜 Credits')
if final.exists(): st.append(f'🎞️ {dur(final):.1f}s')
for s in st: print(f'   {s}')

---
**Every step: automated decisions.** What does that mean?

---
*ROTBOTS — LI-MA TDA 2026, Amsterdam*